In [1]:
import os
from pathlib import Path

import polars as pl
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
import mllabs

data_path = Path('data')

2026-07-05 00:32:06.738112: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-05 00:32:06.771048: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-07-05 00:32:07.456674: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
/home/sun9sun9/python312/lib/python3.12/site-packages/keras/src/export/tf2onnx_lib.py:8: Fu

In [2]:
from mllabs.processor import PolarsLoader
ploader = PolarsLoader(predefined_types={'id': pl.Int64, 'obj_id': pl.Float64}, infer_schema_length=10000)
ploader.fit([data_path / 'train.csv', data_path / 'test.csv', data_path / 'star_classification.csv'])
df_train = ploader.transform([data_path / 'train.csv'])
df_test = ploader.transform([data_path / 'test.csv']).with_columns(
   pl.lit('').alias('class')
)

In [3]:
from itertools import combinations
from mllabs.processor import ExprProcessor
expr_dict1 = {
    'u': pl.when(pl.col('u') < 10).then(
        pl.when(pl.col("class") == "STAR").then(pl.col("u")).otherwise(None).mean()
    ).otherwise(pl.col('u')),
    'alpha90': pl.col('alpha') + 90,
    'spectral_type_galaxy_population': (pl.col('spectral_type').cast(pl.String) + '_' + pl.col('galaxy_population').cast(pl.String)).cast(pl.Categorical)
}
X_mags = ['u', 'g', 'r', 'i', 'z']
expr_dict2 = {
    'mag_mean': pl.mean_horizontal(*X_mags),
    'mag_std': pl.concat_list(X_mags).list.std(),
    'mag_min': pl.min_horizontal(*X_mags),
    'mag_max': pl.max_horizontal(*X_mags),
    'mag_range': pl.max_horizontal(*X_mags) - pl.min_horizontal(*X_mags),
}
X_mags_stat = list(expr_dict2.keys())
expr_dict2 = {
    **expr_dict2,
    'mag_vmax': pl.struct(X_mags).map_elements(
        lambda x: max(x, key=x.get),
        return_dtype=pl.String),
    'redshift_log': (pl.col('redshift') + 1e-1).log(),
    'redshift_1e-4': (pl.col('redshift') == 0.0001).cast(pl.Int8),
    'spectral_type_ord': pl.col('spectral_type').replace({'M': 0, 'G/K': 1, 'A/F': 2, 'O/B': 3}).to_physical().cast(pl.Int8),
    'galaxy_population_i': pl.when(pl.col('galaxy_population') == 'Red_Sequence').then(1).otherwise(0).cast(pl.Int8),
}
X_diff = list()
X_diff = list()
for i, j in combinations(X_mags, 2):
    X_diff.append(f'{i}_{j}')
    expr_dict2[X_diff[-1]] = pl.col(i) - pl.col(j)

X_mags_log = list()
for i in X_mags:
    X_mags_log.append(f'{i}_log')
    expr_dict2[X_mags_log[-1]] = pl.col(i).log()

In [4]:
from sklearn.pipeline import make_pipeline
expr_p = make_pipeline(ExprProcessor(expr_dict1), ExprProcessor(expr_dict2))
df_train = expr_p.fit_transform(df_train)
df_test = expr_p.transform(df_test)

In [5]:
import pickle as pkl
if not os.path.exists('data/lof.pkl'):
    from sklearn.neighbors import LocalOutlierFactor
    from sklearn.cluster import KMeans   
    df_lof = pl.concat([df_train[['alpha90', 'delta']], df_test[['alpha90', 'delta']]])
    lof = LocalOutlierFactor()
    lof.fit(df_lof)
    lof_ = lof.negative_outlier_factor_
    clu_kmeans = KMeans(3000)
    clu_kmeans.fit(df_lof)
    km3000 = clu_kmeans.labels_
    with open('data/lof.pkl', 'wb') as f:
        pkl.dump((lof_, km3000), f)
else:
    with open('data/lof.pkl', 'rb') as f:
        lof_, km3000 = pkl.load(f)
    
df_train = df_train.with_columns(
    pl.Series('lof', lof_[:len(df_train)]),
    pl.Series('km3000', km3000[:len(df_train)], dtype=pl.String).cast(pl.Categorical)
)
df_test = df_test.with_columns(
    pl.Series('lof', lof_[len(df_train):]),
    pl.Series('km3000', km3000[len(df_train):], dtype=pl.String).cast(pl.Categorical)
)
df_train.head()

id,alpha,delta,u,g,r,i,z,redshift,spectral_type,galaxy_population,class,alpha90,spectral_type_galaxy_population,mag_mean,mag_std,mag_min,mag_max,mag_range,mag_vmax,redshift_log,redshift_1e-4,spectral_type_ord,galaxy_population_i,u_g,u_r,u_i,u_z,g_r,g_i,g_z,r_i,r_z,i_z,u_log,g_log,r_log,i_log,z_log,lof,km3000
i64,f32,f32,f32,f32,f32,f32,f32,f32,cat,cat,cat,f32,cat,f32,f32,f32,f32,f32,str,f32,i8,i8,i8,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,cat
0,147.734253,16.959272,25.472122,21.895559,20.357925,19.257113,18.621058,0.408982,"""M""","""Red_Sequence""","""GALAXY""",237.734253,"""M_Red_Sequence""",21.120754,2.731221,18.621058,25.472122,6.851065,"""u""",-0.675342,0,0,1,3.576563,5.114197,6.21501,6.851065,1.537634,2.638447,3.274502,1.100813,1.736868,0.636055,3.237585,3.086284,3.01347,2.957881,2.924293,-1.265492,"""1763"""
1,127.988678,32.346718,20.778509,19.087063,17.587208,17.226067,16.786432,0.157976,"""M""","""Red_Sequence""","""GALAXY""",217.988678,"""M_Red_Sequence""",18.293055,1.636653,16.786432,20.778509,3.992077,"""u""",-1.35489,0,0,1,1.691446,3.191301,3.552443,3.992077,1.499855,1.860996,2.300631,0.361141,0.800776,0.439634,3.03392,2.949011,2.867172,2.846424,2.820571,-1.085606,"""1223"""
2,179.792648,35.344845,21.035202,21.079128,21.171841,20.58263,20.557365,2.82377,"""O/B""","""Blue_Cloud""","""QSO""",269.792664,"""O/B_Blue_Cloud""",20.885233,0.292103,20.557365,21.171841,0.614475,"""r""",1.072874,0,3,0,-0.043926,-0.136639,0.452572,0.477837,-0.092712,0.496498,0.521763,0.589211,0.614475,0.025265,3.046198,3.048284,3.052672,3.024448,3.02322,-1.186385,"""619"""
3,225.818298,48.56942,23.305056,21.050735,19.017754,18.365658,17.914951,0.536099,"""M""","""Red_Sequence""","""GALAXY""",315.818298,"""M_Red_Sequence""",19.93083,2.235331,17.914951,23.305056,5.390104,"""u""",-0.452402,0,0,1,2.25432,4.287302,4.939398,5.390104,2.032982,2.685078,3.135784,0.652096,1.102802,0.450706,3.148671,3.046936,2.945373,2.910483,2.885636,-0.991541,"""1492"""
4,141.836136,19.342852,21.703157,19.47168,18.234449,17.899446,17.616184,0.555761,"""M""","""Red_Sequence""","""GALAXY""",231.836136,"""M_Red_Sequence""",18.984983,1.676354,17.616184,21.703157,4.086973,"""u""",-0.421958,0,0,1,2.231478,3.468708,3.803711,4.086973,1.23723,1.572233,1.855495,0.335003,0.618265,0.283262,3.077458,2.968961,2.903313,2.88477,2.868818,-1.048082,"""1279"""


In [6]:
y2 = 'class'
class_weight = df_train[y2].to_pandas().value_counts().pipe(
    lambda x: x / x.min()
).to_dict()
class_weight

{'GALAXY': 4.56312557419854, 'QSO': 1.4160703060780426, 'STAR': 1.0}

In [7]:
y = 'class_i'
y_repl = {'STAR': 0, 'QSO': 1, 'GALAXY': 2}
y_weight = {'Low': 1.000000, 'Medium': 1.547291, 'High': 17.607549}
df_train = df_train.with_columns(
    **{
        y: pl.col(y2).replace(y_repl).cast(pl.Int8),
        'sample_weight': pl.col(y2).cast(pl.String).replace(class_weight).cast(pl.Float32)
    }
)

In [109]:
X_loc = ['alpha', 'delta']
X_num = ['redshift', 'redshift_log', 'lof', 'alpha90']
X_bin = ['redshift_1e-4', 'galaxy_population_i']
X_nom = ['galaxy_population', 'spectral_type', 'spectral_type_galaxy_population']
X_base = X_loc[1:] + X_num + X_bin + X_nom[1:] + X_diff + X_mags_stat + X_mags_log

X_nom2 = ['km3000']
X_ohe = ['spectral_type', 'spectral_type_galaxy_population']
X_std = ['redshift_log', 'lof'] + X_mags + X_mags_stat + X_mags_log + X_diff

In [110]:
X_num = X_loc + X_num  + X_mags+ X_mags_stat + X_mags_log + X_diff 
X_bin = ['redshift_1e-4', 'galaxy_population_i']
X_nom = ['galaxy_population', 'spectral_type', 'spectral_type_galaxy_population', 'km3000']

In [111]:
df_train[X_std]

redshift_log,lof,u,g,r,i,z,mag_mean,mag_std,mag_min,mag_max,mag_range,u_log,g_log,r_log,i_log,z_log,u_g,u_r,u_i,u_z,g_r,g_i,g_z,r_i,r_z,i_z
f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32
-0.675342,-1.265492,25.472122,21.895559,20.357925,19.257113,18.621058,21.120754,2.731221,18.621058,25.472122,6.851065,3.237585,3.086284,3.01347,2.957881,2.924293,3.576563,5.114197,6.21501,6.851065,1.537634,2.638447,3.274502,1.100813,1.736868,0.636055
-1.35489,-1.085606,20.778509,19.087063,17.587208,17.226067,16.786432,18.293055,1.636653,16.786432,20.778509,3.992077,3.03392,2.949011,2.867172,2.846424,2.820571,1.691446,3.191301,3.552443,3.992077,1.499855,1.860996,2.300631,0.361141,0.800776,0.439634
1.072874,-1.186385,21.035202,21.079128,21.171841,20.58263,20.557365,20.885233,0.292103,20.557365,21.171841,0.614475,3.046198,3.048284,3.052672,3.024448,3.02322,-0.043926,-0.136639,0.452572,0.477837,-0.092712,0.496498,0.521763,0.589211,0.614475,0.025265
-0.452402,-0.991541,23.305056,21.050735,19.017754,18.365658,17.914951,19.93083,2.235331,17.914951,23.305056,5.390104,3.148671,3.046936,2.945373,2.910483,2.885636,2.25432,4.287302,4.939398,5.390104,2.032982,2.685078,3.135784,0.652096,1.102802,0.450706
-0.421958,-1.048082,21.703157,19.47168,18.234449,17.899446,17.616184,18.984983,1.676354,17.616184,21.703157,4.086973,3.077458,2.968961,2.903313,2.88477,2.868818,2.231478,3.468708,3.803711,4.086973,1.23723,1.572233,1.855495,0.335003,0.618265,0.283262
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
-0.491801,-1.009542,20.82873,18.8542,17.703108,17.190536,16.551355,18.225586,1.682177,16.551355,20.82873,4.277374,3.036334,2.936736,2.87374,2.844359,2.806468,1.974529,3.125622,3.638193,4.277374,1.151093,1.663664,2.302845,0.512571,1.151752,0.639181
-0.276295,-1.115867,23.734743,22.359173,20.697865,19.180264,18.947275,20.983866,2.057989,18.947275,23.734743,4.787468,3.16694,3.107237,3.030031,2.953882,2.94166,1.37557,3.036879,4.55448,4.787468,1.661308,3.178909,3.411898,1.517601,1.750589,0.232988
-0.741618,-1.030707,21.94425,21.215857,19.025967,18.772276,18.203396,19.832348,1.643298,18.203396,21.94425,3.740854,3.088506,3.054749,2.945805,2.932381,2.901608,0.728394,2.918283,3.171974,3.740854,2.18989,2.443581,3.012461,0.253691,0.822571,0.56888


In [112]:
X_ohe = ['spectral_type', 'spectral_type_galaxy_population']
X_std = ['redshift_log', 'lof'] + X_mags + X_mags_stat + X_mags_log + X_diff

In [113]:
X_num

['alpha',
 'delta',
 'redshift',
 'redshift_log',
 'lof',
 'alpha90',
 'u',
 'g',
 'r',
 'i',
 'z',
 'mag_mean',
 'mag_std',
 'mag_min',
 'mag_max',
 'mag_range',
 'u_log',
 'g_log',
 'r_log',
 'i_log',
 'z_log',
 'u_g',
 'u_r',
 'u_i',
 'u_z',
 'g_r',
 'g_i',
 'g_z',
 'r_i',
 'r_z',
 'i_z']

In [114]:
from mllabs import Experimenter, ColSelector, Connector, MetricCollector
from mllabs import Pipeline
from mllabs.adapter import XGBoostAdapter, LightGBMAdapter, CatBoostAdapter
from mllabs.nn import NNClassifier
from mllabs import ProgressSessionLogger, TqdmProgressSession
from mllabs.col import ohe_drop_first
from mllabs.collector import SHAPCollector, ModelAttrCollector
from mllabs.filter import RandomFilter
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import balanced_accuracy_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from mllabs.processor import CatOOVFilter
from sklearn.preprocessing import TargetEncoder

In [115]:
!rm exp/phase2_pipeline.db

In [116]:
!rm -rf exp/phase2

In [117]:
p = Pipeline(path='exp', name='phase2_pipeline')

In [118]:
p.set_datasource({
    **{i: 'numerical' for i in X_num},
    **{i: 'binary' for i in X_bin},
    **{i: 'nominal' for i in X_nom+ [y, y2]},
}, targets=[y, y2])

'update'

In [120]:
import lightgbm as lgb
import xgboost as xgb
import catboost as cb
y_edges = {'y': [(None, [y])]}

p.set_grp('pre', role = 'stage', method='transform')
p.set_grp('pre_ft', role = 'stage', method='fit_transform', edges = y_edges)
p.set_grp('pre_ft2', role = 'stage', method='fit_transform', edges = {'y': [(None, [y2])]})

p.set_node('std', grp='pre', processor='sklearn.preprocessing.StandardScaler', edges={'X': [(None, X_std)]})
p.set_node('ohe', grp='pre', processor='sklearn.preprocessing.OneHotEncoder', edges={'X': [(None, X_ohe)]}, params={'sparse_output': False})
p.set_node('coov', processor = CatOOVFilter, grp = 'pre', edges = {'X': [(None, X_nom)]})
p.set_grp('clf', method = 'predict', role = 'head', edges = y_edges)
p.set_grp(
    'xgb', parent = 'clf', processor='xgboost.XGBClassifier', adapter=XGBoostAdapter(eval_mode='both'), 
    params={'random_state': 123, 'n_estimators': 10000, 'enable_categorical': True, 'early_stopping_rounds': 50, 'eval_metric': 'mlogloss'})
p.set_grp(
        'lgb', parent = 'clf', processor=lgb.LGBMClassifier, adapter=LightGBMAdapter(eval_mode='both'), 
        params={'random_state': 123, 'n_estimators': 10000, 'verbose': -1, 'early_stopping': {'stopping_rounds': 50, 'first_metric_only': True},'eval_metric': 'multi_logloss'})
p.set_grp(
    'cb', parent = 'clf', processor=cb.CatBoostClassifier, adapter=CatBoostAdapter(eval_mode='valid'),
    params={'early_stopping_rounds': 50, 'eval_metric': 'AUC', 'verbose': 0, 'random_state': 123, 'cat_features': ColSelector(col_type='category')})
p.set_grp('nn', parent = 'clf', processor = NNClassifier, params = {'metrics': ['sparse_categorical_crossentropy'], 'early_stopping': 10})
p.set_grp('lr', parent='clf', processor=LogisticRegression)
p.set_grp('dt', parent='clf', processor=DecisionTreeClassifier, params={'random_state': 123})
p.set_node('tgt_km3000', grp = 'pre_ft2', processor=TargetEncoder, edges = {'X': [(None, X_nom2)]}, params = {'target_type': 'multiclass'})

p.set_node('xgb1', grp='xgb', edges={'X': [(None,  X_num), ('coov', None)]}, 
           params={'n_estimators': 10000})
p.set_node('lgb1', grp='lgb', edges={'X': [(None, X_base)]}, params={'n_estimators': 10000, 'gpu': None})
p.set_node('cb1', grp='cb', edges={'X': [(None, X_base)]}, params={'n_estimators': 10000})
p.set_node('nn1', grp='nn', edges={'X': [('std', None), ('ohe', ohe_drop_first)]}, params={'epochs': 200})
p.set_node('lr1', grp='lr', edges={'X': [('std', None), ('ohe', ohe_drop_first)]})

{'result': 'new',
 'affected_nodes': [],
 'old_obj': None,
 'obj': <mllabs._pipeline.PipelineNode at 0x741d2d3f7290>}

In [121]:
node = p.get_node('xgb1')
grp = p.get_grp(node.grp)
node.get_attrs(grp)['edges']

{'X': [(None,
   ['alpha',
    'delta',
    'redshift',
    'redshift_log',
    'lof',
    'alpha90',
    'u',
    'g',
    'r',
    'i',
    'z',
    'mag_mean',
    'mag_std',
    'mag_min',
    'mag_max',
    'mag_range',
    'u_log',
    'g_log',
    'r_log',
    'i_log',
    'z_log',
    'u_g',
    'u_r',
    'u_i',
    'u_z',
    'g_r',
    'g_i',
    'g_z',
    'r_i',
    'r_z',
    'i_z']),
  ('coov', None)],
 'y': [(None, ['class_i'])]}

In [123]:
if not os.path.exists('exp/phase2'):
    sss1 = StratifiedShuffleSplit(n_splits = 1, random_state = 123, train_size = 0.8)
    sss2 = StratifiedShuffleSplit(n_splits = 1, random_state = 123, train_size = 0.9)
    e = Experimenter(
        df_train, sp = sss1, sp_v = sss2, splitter_params={'y': y}, path='exp/phase2',
        logger=ProgressSessionLogger(level=['info', 'progress'], session_cls=TqdmProgressSession)
    )   
    e.add_collector(ModelAttrCollector('xgb_evals_results', Connector(processor=xgb.XGBClassifier), 'evals_result'))
    e.add_collector(
        ModelAttrCollector('lgb_evals_results', Connector(processor=lgb.LGBMClassifier), 'evals_result'))
    e.add_collector(
        ModelAttrCollector('cb_evals_results', Connector('_base$', processor=cb.CatBoostClassifier), 'evals_result'))
    e.add_collector(
        ModelAttrCollector('nn_evals', Connector(processor=NNClassifier), result_key='evals_result'))
    e.add_collector(
        MetricCollector('bAcc', Connector(edges = y_edges, role = 'head'), slice(-1, None), balanced_accuracy_score, include_train = True))
    e.add_collector(
        ModelAttrCollector('lgb_feature_importance', Connector(processor=lgb.LGBMClassifier, edges=y_edges), 'feature_importances'))
    e.add_collector(
        ModelAttrCollector(
            'xgb_feature_importance_gain', Connector(processor=xgb.XGBClassifier, edges=y_edges), 'feature_importances', params = {'importance_type': 'gain'}))
    e.add_collector(
        ModelAttrCollector(
            'xgb_feature_importance_cover', Connector(processor=xgb.XGBClassifier, edges=y_edges), 'feature_importances', params = {'importance_type': 'cover'}))
    e.add_collector(
        ModelAttrCollector('cb_feature_importance', Connector(processor=cb.CatBoostClassifier, edges=y_edges), 'feature_importances_pvc'))
    e.add_collector(
        ModelAttrCollector('cb_interaction', Connector(processor=cb.CatBoostClassifier, edges=y_edges), 'feature_importances_interaction'))
    e.add_collector(
        ModelAttrCollector('lr_coef', Connector(processor=LogisticRegression, edges=y_edges), 'coef'))
else:
    e = Experimenter.load('exp/phase2', df_train, logger=ProgressSessionLogger(level=['info', 'progress'], session_cls=TqdmProgressSession))

📁 Created directory: exp/phase2


In [124]:
e.build(p)

Building 4 node(s)


  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

Build complete: 4 node(s)


In [127]:
e.show_error_nodes()

No error nodes found


In [128]:
e.exp(p, nodes = 'xgb1', n_jobs=2, gpu_id_list=[0])

No head nodes to experiment


In [59]:
p.get_node_names('cb1')

['cb1']

In [129]:
e.get_collector('bAcc').get_metrics_agg(None)[0]

,test,train,valid
xgb1,0.951769,0.975166,0.951248


In [ ]:
p.set_node('xgb2', grp='xgb', edges={'X': [(None, X_loc[1:] + X_num + X_bin + X_diff + X_mags_stat + X_mags_log), ('coov', None), ('tgt_km3000', None)]}, 
           params={'n_estimators': 10000})
p.set_node('lgb2', grp='lgb', edges={'X': [(None, X_base), ('tgt_km3000', None)]}, params={'n_estimators': 10000, 'gpu': None})
p.set_node('cb2', grp='cb', edges={'X': [(None, X_base), ('tgt_km3000', None)]}, params={'n_estimators': 10000})
p.set_node('nn2', grp='nn', edges={'X': [('std', None), ('ohe', ohe_drop_first), ('tgt_km3000', None)]}, params={'epochs': 200})
p.set_node('lr2', grp='lr', edges={'X': [('std', None), ('ohe', ohe_drop_first), ('tgt_km3000', None)]})
e.exp(n_jobs=2, gpu_id_list=[0])

In [52]:
e.get_collector('bAcc').get_metrics_agg(None)[0].sort_values('test', ascending = False)

,test,train,valid
lgb1,0.950696,0.956267,0.950954
nn1,0.930930,0.932724,0.929236
lr1,0.898482,0.899930,0.897325


In [25]:
e.get_collector('lgb_feature_importance').get_attrs_agg('lgb2').sort_values(ascending = False).iloc[:10]

delta                        641.0
redshift                     608.0
redshift_log                 536.0
alpha90                      511.0
tgt_km3000__km3000_GALAXY    410.0
g_z                          352.0
tgt_km3000__km3000_STAR      334.0
g_log                        334.0
u_g                          319.0
r_log                        283.0
dtype: float64

In [27]:
e.get_collector('xgb_feature_importance_gain').get_attrs_agg('xgb2').sort_values(ascending = False).iloc[:10]

g_z                          481.291046
redshift                     298.029480
redshift_1e-4                147.961014
u_i                           93.040291
mag_std                       87.927696
g_i                           48.269653
g_log                         42.826180
tgt_km3000__km3000_GALAXY     37.177986
z_log                         34.773258
tgt_km3000__km3000_STAR       32.696632
dtype: float64

In [46]:
e.get_collector('lr_coef').get_attrs_agg('lr2').sort_values(ascending = False).iloc[:10]

2  std__u_log                 6.264473
1  std__r_log                 5.804832
   std__z_log                 5.558737
   std__i_log                 5.287945
   std__g_log                 4.002740
0  std__z_log                 3.404662
2  std__z                     2.343073
   std__i                     2.270111
0  tgt_km3000__km3000_STAR    2.243592
2  std__r                     2.085649
dtype: float64

In [51]:
e.get_collector('cb_interaction').get_attrs_agg('cb2').sort_values(ascending = False).iloc[:20]

feat1          feat2                    
redshift_log   g_z                          6.017913
               tgt_km3000__km3000_GALAXY    4.906412
g_z            tgt_km3000__km3000_GALAXY    4.462546
redshift       g_z                          4.418489
redshift_log   u_i                          4.242768
redshift       tgt_km3000__km3000_GALAXY    4.165511
               redshift_1e-4                3.977335
               redshift_log                 3.018322
redshift_1e-4  g_z                          3.013947
u_i            g_z                          2.988944
redshift_log   redshift_1e-4                2.774923
u_i            tgt_km3000__km3000_GALAXY    2.751668
redshift       tgt_km3000__km3000_STAR      2.710489
redshift_1e-4  tgt_km3000__km3000_GALAXY    2.647240
redshift_log   tgt_km3000__km3000_STAR      2.576597
g_z            tgt_km3000__km3000_STAR      2.045106
redshift       u_i                          1.972798
redshift_1e-4  u_i                          1.874668
redsh

In [ ]:
from sklearn.preprocessing import KBinsDiscretizer 
p.set_node('delta_100', grp = 'pre', processor = KBinsDiscretizer, edges = {'X': [(None, ['delta'])]}, params = {'n_bins': 100, 'encode': 'ordinal'})
p.set_node('delta_500', grp = 'pre', processor = KBinsDiscretizer, edges = {'X': [(None, ['delta'])]}, params = {'n_bins': 500, 'encode': 'ordinal'})
p.set_node('tgt_delta', grp = 'pre_ft2', processor=TargetEncoder, edges = {'X': [('delta_100', None), ('delta_500', None)]}, params = {'target_type': 'multiclass'})
e.build()

In [ ]:
p.set_node('xgb3', grp='xgb', edges={'X': [(None, X_loc[1:] + X_num + X_bin + X_diff + X_mags_stat + X_mags_log), ('coov', None), ('tgt_km3000', None), ('tgt_delta', None)]}, 
           params={'n_estimators': 10000})
p.set_node('lgb3', grp='lgb', edges={'X': [(None, X_base), ('tgt_km3000', None), ('tgt_delta', None)]}, params={'n_estimators': 10000, 'gpu': None})
p.set_node('cb3', grp='cb', edges={'X': [(None, X_base), ('tgt_km3000', None), ('tgt_delta', None)]}, params={'n_estimators': 10000})
p.set_node('nn3', grp='nn', edges={'X': [('std', None), ('ohe', ohe_drop_first), ('tgt_km3000', None), ('tgt_delta', None)]}, params={'epochs': 200})
p.set_node('lr3', grp='lr', edges={'X': [('std', None), ('ohe', ohe_drop_first), ('tgt_km3000', None), ('tgt_delta', None)]})
e.exp(n_jobs=2, gpu_id_list=[0], finalize=True)

In [62]:
e.get_collector('bAcc').get_metrics_agg(None)[0].sort_values('test', ascending = False).iloc[:15]

,test,train,valid
xgb3,0.957715,0.971089,0.957580
cb3,0.957708,0.962739,0.958108
xgb2,0.957261,0.968106,0.957861
lgb3,0.956864,0.961202,0.956684
cb2,0.956850,0.961651,0.956745
lgb2,0.956740,0.960870,0.956747
xgb1,0.953914,0.970428,0.953863
nn3,0.951316,0.952979,0.950653
cb1,0.951201,0.957257,0.950528
lgb1,0.950696,0.956267,0.950954


In [ ]:
from sklearn.preprocessing import KBinsDiscretizer 
p.set_node('alpha_100', grp = 'pre', processor = KBinsDiscretizer, edges = {'X': [(None, ['alpha90'])]}, params = {'n_bins': 100, 'encode': 'ordinal'})
p.set_node('alpha_500', grp = 'pre', processor = KBinsDiscretizer, edges = {'X': [(None, ['alpha90'])]}, params = {'n_bins': 500, 'encode': 'ordinal'})
p.set_node('tgt_alpha', grp = 'pre_ft2', processor=TargetEncoder, edges = {'X': [('alpha_100', None), ('alpha_500', None)]}, params = {'target_type': 'multiclass'})
e.build()

In [ ]:
p.set_node('xgb4', grp='xgb', edges={'X': [(None, X_loc[1:] + X_num + X_bin + X_diff + X_mags_stat + X_mags_log), 
                                           ('coov', None), ('tgt_km3000', None), ('tgt_delta', None), ('tgt_alpha', None)]}, 
           params={'n_estimators': 10000})
p.set_node('lgb4', grp='lgb', edges={'X': [(None, X_base), ('tgt_km3000', None), ('tgt_delta', None), ('tgt_alpha', None)]}, 
           params={'n_estimators': 10000, 'gpu': None})
p.set_node('cb4', grp='cb', edges={'X': [(None, X_base), ('tgt_km3000', None), ('tgt_delta', None), ('tgt_alpha', None)]}, 
           params={'n_estimators': 10000})
p.set_node('nn4', grp='nn', edges={'X': [('std', None), ('ohe', ohe_drop_first), ('tgt_km3000', None), ('tgt_delta', None), ('tgt_alpha', None)]}, 
           params={'epochs': 200})
p.set_node('lr4', grp='lr', edges={'X': [('std', None), ('ohe', ohe_drop_first), ('tgt_km3000', None), ('tgt_delta', None), ('tgt_alpha', None)]})
e.exp(n_jobs=2, gpu_id_list=[0], finalize=True)

In [ ]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from mllabs.processor import CatConverter
from mllabs import ColSelector
p.set_node('cat0', grp='pre', edges={'X': [('alpha_500', None), ('delta_500', None)]}, processor = CatConverter)
p.set_node('cat1', grp='pre', edges={'X': [('alpha_100', None), ('delta_100', None)]}, processor = CatConverter)
p.set_node('lda_mags', grp = 'pre_ft2', processor=LinearDiscriminantAnalysis, edges = {'X': [(None, X_mags)]})
e.build()

In [ ]:
p.set_node('xgb5', grp='xgb', edges={'X': [(None, X_loc[1:] + X_num + X_bin + X_diff + X_mags_stat + X_mags_log), 
                                           ('coov', None), ('tgt_km3000', None), ('tgt_delta', None), ('tgt_alpha', None), ('lda_mags', None)]}, 
           params={'n_estimators': 10000})
p.set_node('lgb5', grp='lgb', edges={'X': [(None, X_base), ('tgt_km3000', None), ('tgt_delta', None), ('tgt_alpha', None), ('lda_mags', None)]}, 
           params={'n_estimators': 10000, 'gpu': None})
p.set_node('cb5', grp='cb', edges={'X': [(None, X_base), ('tgt_km3000', None), ('tgt_delta', None), ('tgt_alpha', None), ('lda_mags', None)]}, 
           params={'n_estimators': 10000})
p.set_node('nn5', grp='nn', edges={'X': [('std', None), ('ohe', ohe_drop_first), ('tgt_km3000', None), ('tgt_delta', None), ('tgt_alpha', None), ('lda_mags', None)]}, 
           params={'epochs': 200})
p.set_node('nn6', grp='nn', 
           edges={'X': [('std', None), ('ohe', ohe_drop_first), ('tgt_km3000', None), ('tgt_delta', None), ('tgt_alpha', None), ('lda_mags', None), 
                        (None, X_nom2), ('cat0', None)]}, 
           params={'epochs': 200, 'cat_cols': ColSelector(col_type = 'category',)})
p.set_node('nn7', grp='nn', 
           edges={'X': [('std', None), ('ohe', ohe_drop_first), ('tgt_km3000', None), ('tgt_delta', None), ('tgt_alpha', None), ('lda_mags', None), 
                        (None, X_nom2), ('cat1', None)]}, 
           params={'epochs': 200, 'cat_cols': ColSelector(col_type = 'category',)})
p.set_node('nn8', grp='nn', 
           edges={'X': [('std', None), ('ohe', ohe_drop_first), ('tgt_km3000', None), ('tgt_delta', None), ('tgt_alpha', None), ('lda_mags', None), 
                        (None, X_nom2)]}, 
           params={'epochs': 200, 'cat_cols': ColSelector(col_type = 'category',)})
p.set_node('nn9', grp='nn', 
           edges={'X': [('std', None), ('ohe', ohe_drop_first), ('tgt_km3000', None), ('tgt_delta', None), ('tgt_alpha', None), ('lda_mags', None), 
                        ('cat1', None)]}, 
           params={'epochs': 200, 'cat_cols': ColSelector(col_type = 'category',)})
e.exp(n_jobs=2, gpu_id_list=[0], finalize=True)

In [122]:
e.get_collector('bAcc').get_metrics_agg(None)[0].sort_values('test', ascending = False)

,test,train,valid
cb5,0.958443,0.963912,0.957949
xgb5,0.958342,0.971035,0.958490
cb4,0.958117,0.963414,0.958295
xgb4,0.958059,0.969408,0.957672
lgb5,0.957972,0.961964,0.957287
xgb3,0.957715,0.971089,0.957580
cb3,0.957708,0.962739,0.958108
lgb4,0.957298,0.961230,0.956512
xgb2,0.957261,0.968106,0.957861
lgb3,0.956864,0.961202,0.956684


In [112]:
e.show_error_nodes()

No error nodes found
